# Рамки и маски: числовая практика

Дополнение к § 3.2. На маленьких координатах проверяем порог, площадь и перекрытие. Для запуска нужен NumPy; затем примените готовую модель из статьи к своему фото.

## Отбор рамок
Поменяйте порог и проследите, какие индексы исчезают.

In [ ]:
import numpy as np
boxes = np.array([[40,30,180,160],[190,40,226,150],[42,31,178,159]])
scores = np.array([0.84, 0.31, 0.73])
for threshold in (0.30, 0.50, 0.80):
    print(threshold, np.flatnonzero(scores >= threshold).tolist())
assert (boxes[:,2] > boxes[:,0]).all()

## Перекрытие
Сравните первую и третью рамки. При одинаковых рамках функция должна вернуть 1, при далёких — 0.

In [ ]:
def iou(a, b):
    left = max(a[0], b[0]); top = max(a[1], b[1])
    right = min(a[2], b[2]); bottom = min(a[3], b[3])
    inter = max(0, right-left) * max(0, bottom-top)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter)
print(round(iou(boxes[0], boxes[2]), 3))
assert iou(boxes[0], boxes[0]) == 1
assert iou(boxes[0], [1000,1000,1010,1010]) == 0

## Пиксели маски
Почему число истинных пикселей меняется с порогом?

In [ ]:
mask_scores = np.array([[0.1,0.7,0.8],[0.2,0.6,0.3]])
for threshold in (0.5,0.75):
    print(threshold, int((mask_scores >= threshold).sum()))

## Опыт с собственной фотографией

Сделайте снимок с двумя предметами, метки которых встречаются в COCO. Перед запуском запишите, что и где ожидаете увидеть. Установите зависимости статьи, замените имя файла и включите run_model. Модель скачивает веса и может выполняться на CPU несколько минут.


In [ ]:
run_model = False
if run_model:
    from pathlib import Path
    import pandas as pd
    from PIL import Image
    import torch
    from torchvision.transforms.functional import to_tensor
    from torchvision.models.detection import (
        FasterRCNN_ResNet50_FPN_V2_Weights,
        fasterrcnn_resnet50_fpn_v2,
    )
    photo = "two_objects.jpg"
    assert Path(photo).is_file()
    image = Image.open(photo).convert("RGB")
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    detector = fasterrcnn_resnet50_fpn_v2(weights=weights).eval()
    with torch.inference_mode():
        result = detector([to_tensor(image)])[0]
    names = weights.meta["categories"]
    report = pd.DataFrame([
        {"class": names[int(label)], "score": float(score),
         "box": [round(x) for x in box.tolist()]}
        for box, label, score in zip(
            result["boxes"], result["labels"], result["scores"]
        )
    ])
    print(report.head(10).to_string(index=False))


## Сравнение порогов

Для порогов 0,30 и 0,70 запишите число показанных рамок и проверьте их на исходном фото. Отдельно отметьте пропущенный предмет и ложную рамку. В статье показано, как наложить прямоугольники; примените этот код к своей фотографии. Затем запустите Mask R-CNN и сравните одну маску с рамкой.


In [ ]:
if run_model:
    for threshold in (0.30, 0.70):
        selected = report[report["score"] >= threshold]
        print("Порог:", threshold, "рамок:", len(selected))
        print(selected[["class", "score", "box"]].head(10).to_string(index=False))


## Самопроверка




### 1. Что возвращает детектор для одного найденного объекта?

A. Только один класс всего кадра.

B. Класс, оценку и рамку.

C. Только число пикселей.

D. Новые веса сети.




### 2. Какова ширина рамки (40,30,180,160)?

A. 120.

B. 130.

C. 140.

D. 180.




### 3. Где находится начало координат изображения?

A. В центре.

B. В левом верхнем углу.

C. В правом нижнем углу.

D. Вне изображения.




### 4. Что произойдёт при повышении порога оценки?

A. Обычно покажется меньше предсказаний.

B. Модель переобучится.

C. Пиксели станут ярче.

D. Увеличится набор классов.




### 5. Что означает отсутствие строк вывода при пороге 0,7?

A. На фото нет предметов.

B. Весов нет.

C. Нет предсказаний выше порога.

D. Все маски пусты.




### 6. Для чего используют IoU?

A. Для измерения перекрытия рамок.

B. Для изменения цвета.

C. Для перевода меток.

D. Для сохранения весов.




### 7. Чем маска отличается от рамки?

A. Маска даёт принадлежность по пикселям.

B. Маска всегда квадратная.

C. Маска меняет класс.

D. Маска не зависит от изображения.




### 8. Почему в model([tensor]) вход заключён в список?

A. Модель ожидает набор изображений.

B. Список обновляет веса.

C. Так создаётся маска.

D. Иначе исчезают классы.




### 9. Какие два порога используются в примере с масками?

A. Цвета и размера.

B. Оценки объекта и значения пикселя маски.

C. IoU и числа классов.

D. Ширины и высоты.




### 10. Можно ли назвать ложную рамку доказательством полной непригодности модели?

A. Да.

B. Только для COCO.

C. Нет, нужна проверка на наборе снимков.

D. Только если оценка 1.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Вспомните три поля учебной таблицы.



**Решение**


**B.** Детектор сообщает категорию, оценку и положение объекта.



### Вопрос 2

**Подсказка**


Вычтите левую координату из правой.



**Решение**


**C.** 180 − 40 = 140 пикселей.



### Вопрос 3

**Подсказка**


Сравните направление роста x и y.



**Решение**


**B.** Обе координаты начинаются в левом верхнем углу.



### Вопрос 4

**Подсказка**


Порог фильтрует готовые кандидаты.



**Решение**


**A.** Предсказания с оценкой ниже нового порога скрываются.



### Вопрос 5

**Подсказка**


Модель могла выдать оценки 0,6 и ниже.



**Решение**


**C.** Вывод ничего не говорит о фактическом отсутствии объектов.



### Вопрос 6

**Подсказка**


Это отношение площади пересечения к площади объединения.



**Решение**


**A.** IoU равен нулю без пересечения и единице для одинаковых рамок.



### Вопрос 7

**Подсказка**


Рамка задаёт четыре координаты.



**Решение**


**A.** Маска описывает форму объекта на сетке пикселей.



### Вопрос 8

**Подсказка**


Вспомните, как обработать несколько кадров одним вызовом.



**Решение**


**A.** Детектор принимает список тензоров изображений.



### Вопрос 9

**Подсказка**


Один фильтрует объекты, другой переводит маску в двоичную.



**Решение**


**B.** У порогов разные входные числа и смысл.



### Вопрос 10

**Подсказка**


Один пример показывает ошибку, но не её частоту.



**Решение**


**C.** Качество оценивают на нескольких размеченных примерах с понятной выборкой.



